# Divergence from the base model for the other training seeds

**GPU: NVIDIA A100.** Run after `colab_run.ipynb`.

`colab_run.ipynb` measures the Kullback-Leibler (KL) divergence to the base model only for the adapters of seed 42. This notebook measures it for the adapters of the other training seeds of the primary model (43 and 44), on the same 200 test items and the same grids, so that the comparison of the two forms at equal divergence covers the three seeds. The own-task scores of these adapters already exist and are not computed again: `scripts/eval_task.py` skips what is saved and only adds the `kl__logit.csv` and `kl__weight.csv` files.

**Resumable**: after a disconnection, run all cells again. Then run `colab_analysis.ipynb`.


## 0. Setup


In [ ]:
import os, subprocess, sys
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU or 'none')
assert GPU, 'No GPU: Runtime > Change runtime type'
EXPECTED = ('A100',)
ALLOW_OTHER_GPU = False   # set to True to run on another GPU anyway (it is recorded in run_info.json)
if not any(g in GPU for g in EXPECTED) and not ALLOW_OTHER_GPU:
    raise SystemExit(f'This notebook is planned for {EXPECTED} (bfloat16, 40 GB or more), got {GPU}: Runtime > Change runtime type')

REPO = '/content/dynalora'
BRANCH = 'paper'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/alexroehrs/dynalora.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log -1 --format='commit %H  %s'
!pip install -q -r requirements.txt
# Colab ships torchao 0.10, and peft 0.21 then refuses to load any LoRA adapter (it requires torchao > 0.16).
# Nothing in this repository uses torchao, so it is removed.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dynalora_results'   # results root (Google Drive); results are not kept in the repository
L4 = f'{RESULTS}/level4'
os.makedirs(L4, exist_ok=True)
sys.path.insert(0, REPO)
from dynalora.runinfo import record_session
e = record_session(f'{L4}/run_info.json', 'KL of the training seeds notebook start', seeds={'adapter_training': 'per adapter (s<seed> in its name)', 'train_subsets': 42, 'test_subsets': 0, 'bootstrap': 0})
print(e['gpu'], e.get('gpu_memory_gb'), e['git'], e['libraries'], sep='\n')
import torch
assert torch.cuda.is_bf16_supported(), 'this GPU has no bfloat16 support'
GRAD_CKPT = '--gradient_checkpointing' if torch.cuda.get_device_properties(0).total_memory < 60 * 2 ** 30 else ''   # 40 GB: checkpointing in training
print('gradient checkpointing in training:', bool(GRAD_CKPT))
FAILED = []   # steps that did not finish; reported at the end

In [ ]:
def sh(cmd, log):
    print(cmd)
    get_ipython().system(f"bash -c 'set -o pipefail; {cmd} 2>&1 | tee -a {log}'")
    return get_ipython().user_ns.get('_exit_code', 1)

PRIMARY = 'Qwen/Qwen2.5-7B-Instruct'
TASKS = ['pubmedqa', 'yelp5', 'gsm8k', 'samsum']
SEEDS = [43, 44]
KL_ITEMS = 200
A_DIR, S_DIR = f'{L4}/adapters', f'{L4}/scores'
tag = PRIMARY.split('/')[-1]

def eval_bs(task):
    return 16 if task in ('gsm8k', 'samsum') else 8


## 1. KL divergence of the adapters of seeds 43 and 44


In [ ]:
for seed in SEEDS:
    for task in TASKS:
        n = f'{tag}__{task}__s{seed}'
        if not os.path.isdir(f'{A_DIR}/{n}'):
            FAILED.append(f'{n}: adapter not found'); print(FAILED[-1]); continue
        rc = sh(f'python scripts/eval_task.py --model {PRIMARY} --task {task} --adapter_dir {A_DIR}/{n} --out_dir {S_DIR}/{n} '
                f'--base_cache {S_DIR}/_base --batch_size {eval_bs(task)} --kl_items {KL_ITEMS}', f'{L4}/kl_seeds.log')
        if rc != 0:
            FAILED.append(f'{n} (exit code {rc})'); print(f'WARNING: {n} did not finish; rerun the notebook to resume')
print('not finished:', FAILED or 'nothing')


## 2. What was written


In [ ]:
import glob, pandas as pd
rows = []
for f in sorted(glob.glob(f'{S_DIR}/{tag}__*__s4[34]/kl__*.csv')):
    k = pd.read_csv(f)
    rows.append(dict(file=f.replace(S_DIR + '/', ''), factors=k.factor.nunique(), items=k.item.nunique() if 'item' in k else len(k)))
print(pd.DataFrame(rows).to_string(index=False) if rows else 'no KL file found')
print('not finished:', FAILED or 'nothing')
